In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

df = pd.read_csv("/content/diabetic_data.csv")

# Map categorical age intervals to their numerical midpoints
age_map = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

features = [
    "age", "num_lab_procedures",
    "num_procedures", "num_medications",
    "number_diagnoses", "number_inpatient"
]

data = df[features + ["time_in_hospital"]].copy()
data = data.dropna()

X = data[features]
y = data["time_in_hospital"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", root_mean_squared_error(y_test, predictions))

MAE: 2.1255618274027297
RMSE: 2.7716772668235703


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


features = [
    "time_in_hospital", "num_lab_procedures",
    "num_procedures", "num_medications",
    "number_diagnoses", "number_inpatient"
]

df = df.dropna(subset=features + ["readmitted"]).copy()

# Convert readmission status into a binary target
df["target"] = (df["readmitted"] == "<30").astype(int)

X = df[features]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.89      0.98      0.93      8792
           1       0.14      0.03      0.05      1139

    accuracy                           0.87      9931
   macro avg       0.51      0.50      0.49      9931
weighted avg       0.80      0.87      0.83      9931



In [9]:

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score



features = [
    "time_in_hospital", "num_medications",
    "number_diagnoses", "number_inpatient",
    "number_emergency", "number_outpatient"
]

X = df[features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = model.fit_predict(X_scaled)

print("Silhouette score:",silhouette_score(X_scaled, clusters))

X_result = X.copy()
X_result["cluster"] = clusters

print(X_result.groupby("cluster").mean().round(2))

Silhouette score: 0.2606715332959082
         time_in_hospital  num_medications  number_diagnoses  \
cluster                                                        
0                    7.62            21.91              8.04   
1                    2.93            11.54              6.16   
2                    4.28            16.11              7.73   

         number_inpatient  number_emergency  number_outpatient  
cluster                                                         
0                    0.55              0.05               0.09  
1                    0.32              0.05               0.08  
2                    2.76              1.03               1.84  
